In [ ]:
# -------------------------------------------------------------------------
# Database connections
# -------------------------------------------------------------------------
# This script connects to two Azure SQL databases:
#
# 1. Content database:
#    Contains publication/release/file metadata such as:
#    - Publications
#    - Releases
#    - Release versions
#    - Data files
#    - Themes
#    - Geographic levels
#
# 2. Statistics database:
#    Contains subject-level statistical metadata such as:
#    - Filters
#    - Filter groups
#    - Filter items / possible filter values
#
# The data from both sources is later combined into a single dataframe that
# can be transformed into search/index-ready JSON documents.

import pandas as pd
from sqlalchemy import create_engine
import urllib
import configparser
from pprint import pprint
import json
import getpass
from tqdm import tqdm
import pyodbc

In [ ]:
# Display installed ODBC data sources.
# Useful for debugging local machine / environment setup.
print(pyodbc.dataSources())

# Display installed ODBC drivers.
# The code below expects "ODBC Driver 18 for SQL Server" to be available.
print(pyodbc.drivers())


In [ ]:
# Read database and Azure OpenAI configuration from config.ini.
# This avoids hard-coding server names, database names, user IDs and API values.

config = configparser.ConfigParser()
config.read('config.ini')

# Connection string for the content database.
# This database holds high-level dataset catalogue information.
content_conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={config['database']['dev_server']};"
    f"DATABASE={config['database']['content_database']};"
    f"Authentication=ActiveDirectoryInteractive;"
    f"TrustServerCertificate=yes;"
    f"Encrypt=yes;"
)

# Connection string for the statistics database.
# This database holds detailed subject/filter/filter-item metadata.
stats_conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={config['database']['dev_server']};"
    f"DATABASE={config['database']['stat_database']};"
    f"Authentication=ActiveDirectoryInteractive;"
    f"TrustServerCertificate=yes;"
    f"Encrypt=yes;"
)

# Create SQLAlchemy engines so pandas can query SQL Server directly.
content_engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(content_conn_str)}"
)

stats_engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(stats_conn_str)}"
)

In [ ]:
# -------------------------------------------------------------------------
# Lookup dictionary for geographic level codes
# -------------------------------------------------------------------------
# The source database stores geographic levels as short codes, for example:
# - LA
# - REG
# - NAT
#
# This dictionary is used later to add human-readable labels to the JSON output.

geography_level_codes = {
    "EDA" : "English devolved area",
    "LA" : "Local authority",
    "LAD" : "Local authority district",
    "LEP" : "Local enterprise partnership",
    "LSIP" : "Local skills improvement plan area",
    "PFA" : "Police force area",
    "INST" : "Institution",
    "MCA" : "Mayoral combined authority",
    "MAT" : "MAT",
    "NAT" : "National",
    "OA" : "Opportunity area",
    "PCON" : "Parliamentary constituency",
    "PROV" : "Provider",
    "REG" : "Regional",
    "RSC" : "RSC region",
    "SCH" : "School",
    "SPON" : "Sponsor",
    "WARD" : "Ward",
    "PA" : "Planning area"
    }

In [ ]:
# -------------------------------------------------------------------------
# Retrieve core content metadata tables
# -------------------------------------------------------------------------
# These queries pull the main metadata required to describe datasets,
# publications, releases and files.

# Files contains records for uploaded files.
# We only need files where Type = 'Data', because these are the actual
# data files that will be represented in the downstream search/index layer.
files_df = pd.read_sql(
    "SELECT * FROM dbo.Files WHERE Type = 'Data'",
    content_engine
)

files_df.head()

# ReleaseFiles links files to specific release versions.
# This table tells us which data files belong to which release version.
release_files_df = pd.read_sql(
    "SELECT * FROM dbo.ReleaseFiles",
    content_engine
)

# ReleaseVersions contains version-level release information.
# We only want versions that:
# - have been approved
# - have a Published value
#
# This avoids indexing draft, unpublished or unapproved release data.
release_versions_df = pd.read_sql(
    """
    SELECT *
    FROM dbo.ReleaseVersions
    WHERE ApprovalStatus = 'Approved'
      AND Published IS NOT NULL
    """,
    content_engine
)

# Publications contains publication-level metadata:
# title, summary, theme, slug, supersession information, etc.
publications_df = pd.read_sql(
    "SELECT * FROM dbo.Publications",
    content_engine
)

# Releases contains release-level metadata.
# Releases are linked to publications and can have multiple release versions.
releases_df = pd.read_sql(
    "SELECT * FROM dbo.Releases",
    content_engine
)

# Themes contains top-level topic/category metadata for publications.
themes_df = pd.read_sql(
    "SELECT * FROM dbo.Themes",
    content_engine
)

# DataSetFileVersionGeographicLevels links dataset file versions to the
# geographic levels they support, such as national, regional or local authority.
geo_levels_df = pd.read_sql(
    "SELECT * FROM dbo.DataSetFileVersionGeographicLevels",
    content_engine
)

In [ ]:
# -------------------------------------------------------------------------
# Identify the latest published version for each release
# -------------------------------------------------------------------------
# A release can have multiple approved/published versions.
# For search/indexing, it is useful to know which records correspond to the
# latest published version so we can mark them as latestData later.

latest_versions_df = (
    release_versions_df[
        (release_versions_df.SoftDeleted == 0) &
        (release_versions_df.Published.notna())
    ]
    .groupby('ReleaseId', as_index=False)["Version"]
    .max()
)

# -------------------------------------------------------------------------
# Build the main dataset metadata dataframe
# -------------------------------------------------------------------------
# The goal of this section is to combine:
# - releases
# - release versions
# - release files
# - physical file metadata
# - publication metadata
# - theme metadata
#
# The resulting dataframe, files_named_df, contains one row per data file /
# release-version combination, enriched with publication and theme context.

# Start with releases and join to release versions.
# This tells us which versions exist for each release.
tdf = releases_df[['Slug', 'Id']].merge(
    release_versions_df,
    left_on='Id',
    right_on='ReleaseId'
)

# Join release-version records to release files.
# This adds the data files associated with each specific release version.
tdf = tdf.merge(
    release_files_df[
        [
            'Name',
            'ReleaseVersionId',
            'FileId',
            'Summary',
            'PublicApiDataSetVersion',
            'PublicApiDataSetId'
        ]
    ],
    left_on='Id_y',
    right_on='ReleaseVersionId'
)

# Join to the Files table.
# This adds physical file metadata and dataset-file identifiers, including:
# - filename
# - content length
# - DataSetFileMeta JSON
# - DataSetFileId
# - SubjectId, which is later used to join to filter metadata
tdf = tdf.merge(
    files_df[
        [
            'Id',
            'Filename',
            'ContentLength',
            'DataSetFileMeta',
            'DataSetFileId',
            'SubjectId'
        ]
    ],
    left_on='FileId',
    right_on='Id'
)

# Drop duplicated ID columns created by merges.
tdf = tdf.drop(['Id_x', 'Id_y'], axis=1)

# Join publication metadata.
# This enriches each dataset file with publication-level context, such as:
# - publication title
# - latest published release version
# - supersession status
# - publication summary
# - theme ID
# - publication slug
tdf = tdf.merge(
    publications_df[
        [
            'Id',
            'Title',
            'LatestPublishedReleaseVersionId',
            'SupersededById',
            'Summary',
            'ThemeId',
            'Slug'
        ]
    ],
    left_on='PublicationId',
    right_on='Id'
)

# Join theme metadata so each dataset can be associated with a topic/theme.
tdf = tdf.merge(
    themes_df[['Id', 'Title']],
    left_on='ThemeId',
    right_on='Id'
)

# Clean up duplicate ID columns and rename the remaining dataset identifier.
tdf = (
    tdf
    .drop('Id_y', axis=1)
    .rename(columns={'Id_x': 'DatasetId'})
)

# Keep only records that are:
# - not soft-deleted
# - published
#
# This ensures downstream indexing does not include unpublished or deleted content.
tdf = tdf[
    (tdf.SoftDeleted == 0) &
    (tdf.Published.notna())
]

# Restrict the dataframe to only the latest published version of each release.
# This is done by joining on ReleaseId and Version against latest_versions_df.
files_named_df = tdf.merge(
    latest_versions_df,
    on=['ReleaseId', 'Version']
)

# Rename ambiguous columns created by merges.
# This makes the dataframe easier to understand and use downstream.
files_named_df = files_named_df.rename(
    columns={
        'Title_x': "PublicationTitle",
        'Title_y': "Theme",
        'Slug_x': 'ReleaseSlug',
        'Slug_y': 'PublicationSlug',
        'Summary_x': 'DatasetSummary',
        'Summary_y': 'PublicationSummary'
    }
)

# Inspect the resulting shape to confirm how many dataset/file records are present.
files_named_df.shape

In [ ]:
# -------------------------------------------------------------------------
# Add geographic level metadata
# -------------------------------------------------------------------------
# A dataset file can support one or more geographic levels.
# The geographic levels are stored in a separate table, so we aggregate them
# into a comma-separated list per file and join them back to files_named_df.

geo_levels_list = (
    files_named_df
    .merge(
        geo_levels_df,
        left_on='FileId',
        right_on='DataSetFileVersionId'
    )
    .groupby('FileId')['GeographicLevel']
    .agg(lambda x: ', '.join(x))
    .reset_index()
)

# Add the aggregated geographic-level codes to each dataset/file row.
files_named_df = files_named_df.merge(
    geo_levels_list,
    on='FileId'
)

In [ ]:
# -------------------------------------------------------------------------
# Retrieve filter metadata from the statistics database
# -------------------------------------------------------------------------
# The statistics database holds the filter structure for each subject.
#
# Relevant tables:
# - Filter:
#   Top-level filter category, e.g. "Characteristic"
#
# - FilterGroup:
#   A group within a filter, e.g. "Gender"
#
# - FilterItem:
#   The possible values within a filter group, e.g. "Male", "Female", "Total"
#
# These are later joined to files_named_df through SubjectId.

filter_df = pd.read_sql(
    "SELECT * FROM dbo.Filter",
    stats_engine
)

filter_group_df = pd.read_sql(
    "SELECT * FROM dbo.FilterGroup",
    stats_engine
)

filter_item_df = pd.read_sql(
    "SELECT * FROM dbo.FilterItem",
    stats_engine
)

In [ ]:
# -------------------------------------------------------------------------
# Build a filter lookup dataframe
# -------------------------------------------------------------------------
# This section converts the normalised filter tables into a flattened structure:
#
# SubjectId | FilterGroupId | Filter Category | Filter Name | Filter Values
#
# This makes it easier to attach filter metadata to each dataset row.

filter_names_df = (
    filter_df
    # Join Filter to FilterGroup.
    # This links each filter category to its groups.
    .merge(
        filter_group_df,
        left_on='Id',
        right_on='FilterId',
        how='inner'
    )
    # Join FilterGroup to FilterItem.
    # This adds the possible values for each filter group.
    .merge(
        filter_item_df,
        left_on='Id_y',
        right_on='FilterGroupId',
        how='inner'
    )
)

# Select only the columns needed downstream.
#
# Id_x           = Filter ID
# Label_x        = Filter category label
# Label_y        = Filter group/filter name
# Label          = Filter item/value label
# SubjectId      = Subject identifier, used to join to dataset files
# FilterGroupId  = Identifier for the filter group
filter_names_df = filter_names_df[
    [
        'Id_x',
        'Label_x',
        'Label_y',
        'Label',
        'SubjectId',
        'FilterGroupId'
    ]
]

# Group filter values so each row represents one filter group.
# Multiple filter item labels are concatenated using '||' so they can be
# split back into lists later when building JSON documents.
filter_names_df = (
    filter_names_df
    .groupby(
        [
            'SubjectId',
            'FilterGroupId',
            'Label_x',
            'Label_y'
        ]
    )[['Label']]
    .agg(lambda x: '||'.join(x))
    .reset_index()
)

# Rename columns to clearer business-friendly names.
filter_names_df = (
    filter_names_df
    .rename(
        columns={
            'Label_x': 'Filter Category',
            'Label_y': 'Filter Name',
            'Label': 'Filter Values'
        }
    )
    .reset_index(drop=True)
)

filter_names_df.head(3)

In [ ]:
# -------------------------------------------------------------------------
# Attach filter metadata to the main dataset dataframe
# -------------------------------------------------------------------------
# Join filter metadata to the dataset metadata using SubjectId.
#
# This creates one row per dataset/filter group combination.
# For example, if a dataset has three filter groups, the dataset may appear
# three times in files_named_df after this merge — once for each filter group.

files_named_df = files_named_df.merge(
    filter_names_df,
    on='SubjectId',
    how='left'
)

## Processing Metadata for Downstream use

In [ ]:
# Looking at one particular row to identify all the useful information
sample_row = files_named_df.iloc[11]
sample_row

In [ ]:
def extract_time_period(row):
    """
    Extracts the human-readable time period range from the DataSetFileMeta JSON field.

    DataSetFileMeta contains a nested TimePeriodRange object with Start and End values.
    This function converts those nested values into a single string such as:
    "Academic year 2020 - Academic year 2023".

    If the input is not a JSON string, it is returned unchanged.
    """
    if not isinstance(row, str):
        return row

    # Parse the JSON metadata stored in the DataSetFileMeta column.
    row_parsed = json.loads(row)

    # Build the start period label from the nested TimePeriodRange structure.
    start = (
        f"{row_parsed['TimePeriodRange']['Start']['TimeIdentifier']['label']} "
        f"{row_parsed['TimePeriodRange']['Start']['Period']}"
    )

    # Build the end period label from the nested TimePeriodRange structure.
    end = (
        f"{row_parsed['TimePeriodRange']['End']['TimeIdentifier']['label']} "
        f"{row_parsed['TimePeriodRange']['End']['Period']}"
    )

    # Combine the start and end values into a single readable range.
    time_period = f"{start} - {end}"

    return time_period

def extract_filters_and_indicators(row):
    """
    Extracts filter labels and indicator labels from the DataSetFileMeta JSON field.

    The metadata contains two important arrays:
    - Filters: dimensions that users can filter by, such as gender, age, phase, etc.
    - Indicators: measurable columns/statistics available in the dataset.

    The labels are joined using ' || ' so they can be stored as compact string metadata
    and later split back into lists when creating JSON search documents.

    If the input is not a JSON string, the original value is returned twice.
    """
    if not isinstance(row, str):
        return row, row

    # Parse the JSON metadata stored in the DataSetFileMeta column.
    row_parsed = json.loads(row)

    # Extract the display labels for all filters in the metadata.
    filters = ' || '.join(x['Label'] for x in row_parsed['Filters'])

    # Extract the display labels for all indicators/statistical columns.
    indicators = ' || '.join(x['Label'] for x in row_parsed['Indicators'])

    return filters, indicators

In [ ]:
files_named_df = files_named_df[~files_named_df.Published.isna()]
files_named_df = files_named_df.reset_index(drop=True)

In [ ]:
def generate_metadata(row):
    """
    Generates a compact JSON metadata object for a dataset row.

    This function pulls key derived metadata from DataSetFileMeta and stores it in a
    simplified structure used later when creating dataset-level search documents.

    The returned JSON contains:
    - time_period: the dataset's available time period range
    - filters: filter labels available for the dataset
    - other_columns: indicator/statistical column labels available for the dataset

    Some additional row values are assigned to variables for readability/context,
    although they are not currently included in the returned metadata object.
    """
    # Extract the dataset's time coverage from the raw JSON metadata.
    time_period = extract_time_period(row['DataSetFileMeta'])

    # Extract available filters and indicators from the raw JSON metadata.
    filters, indicators = extract_filters_and_indicators(row['DataSetFileMeta'])

    # Create a simplified metadata dictionary for downstream JSON generation.
    metadata = {
        'time_period': time_period,
        'filters': filters,
        'other_columns': indicators,
    }

    # Store as JSON so it can be safely persisted in a dataframe column.
    return json.dumps(metadata)

In [ ]:
# For Filter categories that only have one filter, change the filter name to the filter category name instead of 'Default'
files_named_df.loc[files_named_df['Filter Name']=='Default', 'Filter Name'] = files_named_df['Filter Category']

files_named_df['metadata'] = files_named_df.apply(generate_metadata, axis=1)

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")
assert enc.decode(enc.encode("hello world")) == "hello world"

# To get the tokeniser corresponding to a specific model in the OpenAI API:
enc = tiktoken.encoding_for_model("text-embedding-3-large")

### Generate Embeddings with token count

In [ ]:
import os
import time
from openai import AzureOpenAI
import re

def strip_control_chars(text):
    """
    Removes ASCII control characters from text before sending it for embedding.

    Control characters can cause issues when sending strings to embedding models
    or when serialising content for downstream systems.
    """
    return re.sub(r"[\x00-\x1f\x7f-\x9f]", "", text)

def sanitize_for_embedding(text):
    """
    Cleans and normalises text before it is sent to the embedding model.

    The function:
    - returns None for non-string values
    - trims leading/trailing whitespace
    - removes control characters
    - ensures the text is valid UTF-8

    This helps reduce embedding failures caused by malformed or invalid text.
    """
    if not isinstance(text, str):
        return None

    # Remove leading and trailing whitespace.
    text = text.strip()

    # Remove hidden/control characters that may break API requests.
    text = strip_control_chars(text)

    # Re-encode as UTF-8, ignoring invalid byte sequences.
    text = text.encode("utf-8", errors="ignore").decode("utf-8")

    return text

def batch(items, size=20):
    """
    Splits a list into smaller chunks of a fixed size.

    This is used to send embedding requests in batches rather than one item at a time.
    Batching improves throughput and reduces the number of API calls.
    """
    for i in range(0, len(items), size):
        yield items[i:i+size]

def get_embeddings(input_text: str | list[str], model_name: str, dimensions: int = 1536):
    """
    Generates vector embeddings for one text string or a list of text strings using Azure OpenAI.

    For list inputs:
    - texts are processed in batches
    - progress is shown using tqdm
    - total token usage is accumulated
    - failed requests are retried with exponential backoff

    For single string inputs:
    - one embedding request is made directly

    Parameters
    ----------
    input_text:
        Either a single string or a list of strings to embed.

    model_name:
        Name of the Azure OpenAI embedding deployment/model.

    dimensions:
        Number of embedding dimensions to return. Defaults to 1536.

    Returns
    -------
    For list input:
        Tuple of:
        - list of embeddings
        - total tokens used

    For single string input:
        Tuple of:
        - list containing the embedding
        - total tokens used
    """
    # Create an Azure OpenAI client using values from config.ini.
    # The embedding endpoint is formatted dynamically using the model/deployment name.
    client = AzureOpenAI(
        api_key=config['azure_openai']['api_key'],
        base_url=config['azure_openai']['embedding_endpoint'].format(model_name=model_name),
        api_version=config['azure_openai']['api_version']
    )

    # Handle batch embedding generation when a list of input texts is provided.
    if isinstance(input_text, list):
        print('Processing list of inputs...')

        all_embeddings = []
        total_tokens = 0

        # Split the input texts into smaller batches to avoid oversized API requests.
        batches = list(batch(input_text, 15))

        print('Generating embeddings...')

        # Track progress across all input texts.
        with tqdm(total=len(input_text), desc="Embedding texts", unit="text") as pbar:
            for chunk in batches:

                # Retry each batch up to 2 times to handle transient API failures.
                for attempt in range(1, 2 + 1):
                    try:
                        response = client.embeddings.create(
                            model="text-embedding-3-large",  # Azure OpenAI deployment name
                            input=chunk,
                            dimensions=dimensions
                        )

                        # Store returned embedding vectors in the same order as input texts.
                        all_embeddings.extend(d.embedding for d in response.data)

                        # Track token consumption for cost monitoring.
                        total_tokens += response.usage.total_tokens

                        # Update progress by the number of texts processed in this batch.
                        pbar.update(len(chunk))
                        break

                    except Exception as e:
                        # If all retry attempts fail, surface the problematic chunk
                        # and raise a clear error for debugging.
                        if attempt == 2:
                            print(f"Tokens Processed: {total_tokens}")
                            print(chunk)
                            raise RuntimeError(
                                f"Embedding failed after {2} attempts"
                            ) from e

                        # Exponential backoff before retrying transient failures such as
                        # throttling, timeouts, or gateway errors.
                        time.sleep(2 ** attempt)

        return all_embeddings, total_tokens

    # Handle the simpler case where a single text string is provided.
    response = client.embeddings.create(
        input=input_text,
        model=model_name,
        dimensions=dimensions
    )

    # Return the embedding vector and token usage.
    embeddings = [x.embedding for x in response.data]

    return embeddings, response.usage.total_tokens

### Push embeddings to JSON data to push to Azure AI Search

In [ ]:
# For Datasets with no filters, since they have a single entry setting the ID to the FileID instead
files_named_df.loc[files_named_df['Filter Name'].isna(), 'FilterGroupId'] = files_named_df['FileId']

In [ ]:
def truncate_by_token_limit(
    text: str,
    max_tokens: int,
    delimiter: str = "||",
    encoding_name: str = "cl100k_base"
) -> str:
    """
    Truncates a delimiter-separated string so that it stays within a token limit.

    This is mainly used for long filter value lists before they are included in:
    - embedding text
    - JSON search documents

    Instead of cutting text mid-value, this function preserves whole delimiter-separated
    values and stops adding values once the token limit would be exceeded.

    Parameters
    ----------
    text:
        Input string containing delimiter-separated values.

    max_tokens:
        Maximum number of tokens allowed in the returned string.

    delimiter:
        Separator used between values. Defaults to '||'.

    encoding_name:
        Tokeniser encoding name used by tiktoken.

    Returns
    -------
    A truncated string containing only complete values within the token limit.
    """
    # Load the tokeniser used to estimate token count.
    encoder = tiktoken.get_encoding(encoding_name)

    # Split the original text into logical values.
    parts = text.split(delimiter)

    kept_parts = []
    current_token_count = 0

    for part in parts:
        # Rebuild the candidate text with the next part included.
        # This ensures token counting reflects the actual output string.
        candidate = (
            delimiter.join(kept_parts + [part])
            if kept_parts
            else part
        )

        # Count tokens for the candidate text.
        tokens = len(encoder.encode(candidate))

        # Stop before exceeding the token limit.
        if tokens > max_tokens:
            break

        kept_parts.append(part)
        current_token_count = tokens

    # Return the truncated text while preserving the original delimiter style.
    return delimiter.join(kept_parts).strip()

files_named_df['Filter Values'].apply(lambda x: len(enc.encode(x)) if isinstance(x, str) else 0).quantile(0.95)

In [ ]:
filtered_publications = ['02f515a1-c620-4aac-b728-08dedcd6fc40',
'135873c7-2375-4a94-03bb-08deb8011eea',
'240b4a45-da1a-41ba-ef12-08dedb6375ba',
'28438e3e-ac64-46d1-03bd-08deb8011eea',
'2cfc4300-70fb-4500-b727-08dedcd6fc40',
'632da40d-7e9f-47e9-8ffe-08dedb747496',
'78724875-1235-4992-03bc-08deb8011eea',
'81139eb4-8b66-4a9b-b62d-08dedb7678d9',
'83f43ee2-4d3c-420d-b62c-08dedb7678d9',
'96f418e7-3ddb-4a8c-60dc-08deb7f1c424',
'a676364e-1fb2-4676-03ba-08deb8011eea',
'a91d9e05-be82-474c-85ae-4913158406d0',
'b6f5f5f7-4c25-45b8-b722-08dedcd6fc40',
'fb724e25-843f-4caa-b729-08dedcd6fc40']

filtered_release_verions = ['114aa1ad-af8f-4054-1e90-08dedcd6fd54',
'122def7c-1c85-44bd-235d-08deb8015fcb',
'41edff3f-b70c-441e-1e88-08dedcd6fd54',
'8e3530c5-deee-4525-2415-08deb8015fcb',
'a575074b-c3fe-4674-2416-08deb8015fcb',
'b2124684-aa61-4ef7-235e-08deb8015fcb',
'b9011bfa-511c-4a65-1e91-08dedcd6fd54',
'ccab85f1-36e9-42b6-2417-08deb8015fcb',
'cdb1156a-8d58-494e-0b82-08dedb771f7c',
'daadfe3d-67f4-4f40-2414-08deb8015fcb',
'db7aad55-30b2-4d7e-bb45-08dec09602df',
'ee0ac8af-aa87-499c-79c6-08dedb6df4a4',
'ef0e677e-defc-4233-0b83-08dedb771f7c',
'f9f71355-cc70-455f-fa7f-08dedb74caac',
'fa207ac8-0228-4fdf-1e8f-08dedcd6fd54'
]

shortlisted_publications = files_named_df[files_named_df.PublicationId.str.lower().isin(filtered_publications)]
shortlisted_releases = shortlisted_publications[shortlisted_publications.ReleaseVersionId.str.lower().isin(filtered_release_verions)]

files_named_df = shortlisted_releases

In [ ]:
from tqdm import tqdm

output_folder = 'dataset_metadata/shortlisted/filter_level'
os.makedirs(output_folder, exist_ok=True)

def make_embedding_text(row):
    """
    Builds the natural-language text that will be embedded for filter-level search.

    The embedding text includes:
    - dataset name
    - filter name
    - a truncated list of possible filter values

    This helps semantic search retrieve relevant datasets or filter groups based on
    user queries that mention dataset titles, filter names, or filter values.
    """
    # Limit filter values to avoid sending overly long text to the embedding model.
    filter_values = (
        truncate_by_token_limit(row['Filter Values'], 250).split('||')
        if isinstance(row['Filter Values'], str)
        else []
    )

    # Convert structured filter metadata into readable natural language.
    content = (
        f"The name of this dataset is: {row['Name']}\n"
        f"The name of the filter is {row['Filter Name']} and it can take the following possible values:\n"
        f"-{'\n-'.join(filter_values)}"
    )

    return content

texts = files_named_df.apply(make_embedding_text, axis=1).tolist()
embeddings, total_tokens_used = get_embeddings(texts, 'text-embedding-3-large')
print(f"Total tokens used for embedding: {total_tokens_used}, approximate cost: {(total_tokens_used/1000)*0.0001}")
files_named_df['embedding'] = embeddings

for i, row in tqdm(files_named_df.iterrows(), total=len(files_named_df)):
    search_data = dict()
    search_data['fileId'] = row['FileId'].lower()
    search_data['dataSetFileId'] = row['DataSetFileId'].lower()
    search_data['datasetTitle'] = row['Name']

    search_data['publicationId'] = row['PublicationId'].lower()
    search_data['publicationTitle'] = row['PublicationTitle']
    search_data['subjectId'] = row['SubjectId'].lower()

    search_data['filterGroupId'] = row['FilterGroupId'].lower()
    
    search_data['filterCategory'] = row['Filter Category']
    search_data['filterName'] = row['Filter Name']
    search_data['filterValues'] = truncate_by_token_limit(row['Filter Values'], 250).split('||') if isinstance(row['Filter Values'], str) else []
    
    search_data['indicators'] = [x['Label'].strip() for x in json.loads(row['DataSetFileMeta'])['Indicators']]

    search_data['releaseVersionId'] = row['ReleaseVersionId'].lower()
    search_data['latestData'] = row['ReleaseVersionId'] == row['LatestPublishedReleaseVersionId']

    search_data['embeddingText'] = make_embedding_text(row)
    search_data['embedding'] = row['embedding']

    filename = f"{search_data['filterGroupId']}.json"
    filepath = os.path.join(output_folder, filename)

    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(search_data, f, indent=2, ensure_ascii=False)

In [ ]:
from html.parser import HTMLParser

class _HtmlToText(HTMLParser):
    """
    Lightweight HTML-to-text parser used to clean publication and dataset summaries.

    The parser extracts visible text from HTML and preserves links by appending
    the href in brackets after linked text.

    This is useful because summaries may be stored as HTML in the content database,
    while search indexes usually need plain text.
    """

    def __init__(self):
        """
        Initialises an empty text buffer and tracks the current hyperlink, if any.
        """
        super().__init__()
        self._chunks = []
        self._current_href = None

    def handle_starttag(self, tag, attrs):
        """
        Captures href values when an anchor tag starts.

        If the parser encounters an <a> tag, the href is stored temporarily so it
        can be appended next to the link text when handle_data is called.
        """
        if tag == "a":
            for k, v in attrs:
                if k == "href":
                    self._current_href = v

    def handle_data(self, data):
        """
        Captures visible text from the HTML.

        If the text belongs to a hyperlink, the URL is appended in brackets so
        downstream search users still have access to the original link target.
        """
        self._chunks.append(data)

        # Preserve the URL next to the linked text.
        if self._current_href:
            self._chunks.append(f" ({self._current_href})")

    def handle_endtag(self, tag):
        """
        Clears the currently tracked hyperlink when an anchor tag ends.
        """
        if tag == "a":
            self._current_href = None

    def get_text(self):
        """
        Returns the extracted plain-text content as a single string.
        """
        return ' '.join(self._chunks)

def html_to_text(html: str) -> str:
    """
    Converts an HTML string into plain text.

    This is used to clean dataset summaries before placing them into search-ready
    JSON documents.
    """
    parser = _HtmlToText()
    parser.feed(html)
    return parser.get_text()

In [ ]:
def format_bytes(size_bytes: int) -> str:
    """
    Converts a file size in bytes into a human-readable string.

    Examples:
    - 500 -> '500.00 B'
    - 2048 -> '2.00 KB'
    - 1048576 -> '1.00 MB'

    This makes file sizes easier to display in downstream search or UI experiences.
    """
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size_bytes < 1024:
            return f"{size_bytes:.2f} {unit}"

        size_bytes /= 1024

filters_per_dataset = (
    files_named_df
    .groupby('DatasetId')
    .apply(
        lambda x: {}
        if (
            len(x) == 1 and
            x['Filter Values'].isna().iloc[0]
        )
        else dict(zip(x['Filter Name'], 
                      x['Filter Values'].apply(lambda x: truncate_by_token_limit(x, 100))
                      ))
    )
    .reset_index(name='Filters_and_Values')
)

dataset_level_df = files_named_df.merge(filters_per_dataset, on='DatasetId', how='inner').drop_duplicates('DatasetId').reset_index(drop=True)

output_folder = 'dataset_metadata/shortlisted/dataset_level'
os.makedirs(output_folder, exist_ok=True)

for i, row in tqdm(dataset_level_df.iterrows(), total=len(dataset_level_df)):
    data = json.loads(row['metadata'])
    search_data = dict()
    search_data['fileId'] = row['FileId'].lower()
    search_data['dataSetFileId'] = row['DataSetFileId'].lower()
    search_data['filename'] = row['Filename'].split('.')[0]
    search_data['fileSize'] = format_bytes(row['ContentLength'])
    search_data['fileExtension'] = row['Filename'].split('.')[1]

    search_data['title'] = row['Name']
    search_data['content'] = html_to_text(row['DatasetSummary']) if isinstance(row['DatasetSummary'], str) else ''

    search_data['themeId'] = row['ThemeId'].lower()
    search_data['themeTitle'] = row['Theme']

    search_data['publicationId'] = row['PublicationId'].lower()
    search_data['publicationTitle'] = row['PublicationTitle']
    search_data['publicationSlug'] = row['PublicationSlug']
    search_data['subjectId'] = row['SubjectId'].lower()
    search_data['releaseId'] = row['ReleaseId'].lower()
    search_data['releaseTitle'] = row['Name']
    search_data['releaseSlug'] = row['ReleaseSlug']
    search_data['releaseVersionId'] = row['ReleaseVersionId'].lower()

    search_data['latestData'] = row['ReleaseVersionId'] == row['LatestPublishedReleaseVersionId']
    search_data['isSuperseded'] = isinstance(row['SupersededById'], str)

    search_data['published'] = str(row['PublishedDisplayDate'])#.strftime('%d %B %y')
    search_data['lastUpdated'] = str(row['Published'])#.strftime('%d %B %y')

    search_data['api'] = {
        'version' : row['PublicApiDataSetVersion'] if isinstance(row['PublicApiDataSetVersion'], str) else '',
        'id' : row['PublicApiDataSetId'] if isinstance(row['PublicApiDataSetId'], str) else ''
    }

    search_data['numDataFileRows'] = json.loads(row['DataSetFileMeta'])['NumDataFileRows']
    search_data['geographicLevels'] = [x.strip() for x in row['GeographicLevel'].split(',')]
    search_data['geographicLevelsLabels'] = [geography_level_codes[x.strip()] for x in row['GeographicLevel'].split(',')]
    search_data['timePeriodRange'] = {
        "from": data['time_period'].split('-')[0].strip(),
        "to": data['time_period'].split('-')[1].strip()
        }
    search_data['filters'] = [x.strip() for x in data['filters'].split('||')]
    search_data['indicators'] = [x.strip() for x in data['other_columns'].split('||')]
    
    search_data['releaseType'] = row['Type']
    
    filename = f"{search_data['fileId']}.json"
    filepath = os.path.join(output_folder, filename)

    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(search_data, f, indent=2, ensure_ascii=False)